# Project 3 — Transactional Event Processing and Observability

Build a reproducible offline event ingestion service that validates requests, enforces idempotency in SQLite, maintains category aggregates, and emits secret-free structured operational metrics. A companion **installable Python package** provides a CLI and unit tests.

**Advanced portfolio · Suggested time: 5–8 hours · Python 3.11+**

## Requirements

- Validate external JSON and reject invalid amounts, categories, and identifiers.
- Enforce idempotency keys atomically with the database side effects.
- Reject conflicting payloads reusing a key.
- Maintain aggregate totals under SQLite transactions.
- Report accepted, replayed, and rejected outcomes without exposing sensitive data.
- Test failures using in-memory databases and an explicit clock/fake transport boundary.

The notebook uses **only an in-memory SQLite database**, so it does not modify persistent files. The packaged CLI writes to a database file only when the user explicitly runs it with a path.

## 1. Immutable event model and input validation

Define a stable schema before writing storage code. A frozen dataclass prevents ordinary reassignment but does not automatically validate types. Use a validation constructor for untrusted JSON dictionaries. Amounts use integer units for deterministic arithmetic.

In [ ]:
from dataclasses import dataclass
import json

@dataclass(frozen=True)
class Event:
    event_id: str
    category: str
    amount: int

    def __post_init__(self):
        if not isinstance(self.event_id, str) or not self.event_id.strip():
            raise ValueError("event_id must be nonempty text")
        if not isinstance(self.category, str) or not self.category.strip():
            raise ValueError("category must be nonempty text")
        if isinstance(self.amount, bool) or not isinstance(self.amount, int) or self.amount < 0:
            raise ValueError("amount must be a nonnegative integer")

def parse_event(data):
    if not isinstance(data, dict):
        raise ValueError("expected a JSON object")
    try:
        return Event(data["event_id"], data["category"], data["amount"])
    except KeyError as error:
        raise ValueError(f"missing field: {error.args[0]}") from error

print(parse_event({"event_id": "E1", "category": "books", "amount": 5}))
assert parse_event(json.loads('{"event_id":"E2","category":"pens","amount":2}')).amount == 2

**Design question:** Why prefer integer monetary units over floating-point values for precise billing?

## 2. Database schema, uniqueness and connection lifecycle

SQLite constraints enforce the idempotency key's uniqueness even when two callers race. A transaction should apply the logical effect and save the corresponding outcome together. Explicit connection.close remains the caller's responsibility.

In [ ]:
import sqlite3

def initialize(connection):
    connection.execute("""
        CREATE TABLE IF NOT EXISTS processed_events (
            event_id TEXT PRIMARY KEY,
            category TEXT NOT NULL,
            amount INTEGER NOT NULL CHECK (amount >= 0)
        )
    """)
    connection.execute("""
        CREATE TABLE IF NOT EXISTS totals (
            category TEXT PRIMARY KEY,
            total INTEGER NOT NULL CHECK (total >= 0)
        )
    """)
    connection.commit()

connection = sqlite3.connect(":memory:")
initialize(connection)
print(connection.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall())

**Design question:** Why is a UNIQUE/PRIMARY KEY constraint stronger than checking a Python set first?

## 3. Atomic idempotent processing

For a new event ID, insert the event record and increment the corresponding category total in the *same transaction*. A replay with an identical payload returns 'replayed' without incrementing totals. Conflicting reuse of an ID is rejected. The sqlite3 connection's with statement commits or rolls back, but does not close the connection.

In [ ]:
def apply_event(db, event):
    with db:
        previous = db.execute(
            "SELECT category, amount FROM processed_events WHERE event_id = ?",
            (event.event_id,)
        ).fetchone()
        if previous is not None:
            if previous != (event.category, event.amount):
                raise ValueError("idempotency key reused with different payload")
            return "replayed"

        db.execute(
            "INSERT INTO processed_events (event_id, category, amount) VALUES (?, ?, ?)",
            (event.event_id, event.category, event.amount)
        )
        db.execute("""
            INSERT INTO totals (category, total) VALUES (?, ?)
            ON CONFLICT(category) DO UPDATE SET total = total + excluded.total
        """, (event.category, event.amount))
    return "accepted"

event = Event("E1", "books", 5)
print(apply_event(connection, event))
print(apply_event(connection, event))
print(connection.execute("SELECT * FROM totals").fetchall())
assert connection.execute("SELECT total FROM totals WHERE category='books'").fetchone()[0] == 5

**Design question:** What could go wrong if storing the event and updating totals happened in two separate transactions?

## 4. Safe error reporting and operational counters

Public error responses should expose stable codes, not raw SQL statements or stack traces. Observability measures accepted, replayed, and rejected counts, while explicitly avoiding credentials, request payloads, or user-sensitive fields.

In [ ]:
def process_batch(db, raw_events):
    counters = {"accepted": 0, "replayed": 0, "rejected": 0}
    errors = []
    for index, raw in enumerate(raw_events, start=1):
        try:
            event = parse_event(raw)
            status = apply_event(db, event)
            counters[status] += 1
        except (ValueError, TypeError, sqlite3.Error) as error:
            counters["rejected"] += 1
            errors.append({"row": index, "code": "INVALID_EVENT"})
    return {"metrics": counters, "errors": errors}

batch = [
    {"event_id": "E2", "category": "books", "amount": 10},
    {"event_id": "E2", "category": "books", "amount": 10},
    {"event_id": "E3", "category": "pens", "amount": -1},
]
report = process_batch(connection, batch)
print(json.dumps(report, sort_keys=True))
assert report["metrics"] == {"accepted": 1, "replayed": 1, "rejected": 1}

**Design question:** What additional trace and timing fields could be logged without exposing the raw event contents?

## 5. Database consistency and failure-path tests

Test the core invariants: duplicate events have one effect, conflicting keys reject changes, invalid amounts never reach storage, and distinct events aggregate correctly. Tests use independent in-memory connections and close them in tearDown.

In [ ]:
import unittest

class EventProcessorTests(unittest.TestCase):
    def setUp(self):
        self.db = sqlite3.connect(":memory:")
        initialize(self.db)

    def tearDown(self):
        self.db.close()

    def test_replay(self):
        event = Event("K1", "books", 7)
        self.assertEqual(apply_event(self.db, event), "accepted")
        self.assertEqual(apply_event(self.db, event), "replayed")
        self.assertEqual(self.db.execute("SELECT total FROM totals WHERE category='books'").fetchone()[0], 7)

    def test_conflict(self):
        apply_event(self.db, Event("K1", "books", 7))
        with self.assertRaises(ValueError):
            apply_event(self.db, Event("K1", "books", 9))
        self.assertEqual(self.db.execute("SELECT total FROM totals WHERE category='books'").fetchone()[0], 7)

    def test_invalid(self):
        with self.assertRaises(ValueError):
            Event("X", "books", -1)

    def test_aggregation(self):
        apply_event(self.db, Event("K1", "books", 2))
        apply_event(self.db, Event("K2", "books", 3))
        self.assertEqual(self.db.execute("SELECT total FROM totals WHERE category='books'").fetchone()[0], 5)

result = unittest.TextTestRunner(verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(EventProcessorTests))
assert result.wasSuccessful()
connection.close()

**Design question:** Why should integration tests use the actual storage engine when evaluating database atomicity?

## 6. Architecture review and boundaries

The package in `event_processor_package/` moves these operations into reusable modules and provides an argparse CLI with an explicit database path. The separation is:

- **Domain:** Event validation and invariants.
- **Storage:** SQLite schema and transactional processing.
- **Interface:** CLI accepts JSON and renders safe statuses.
- **Tests:** In-memory SQLite proves idempotency and rejection behavior.

A production system would also need durable queues, access controls, timeouts, monitoring, operational retention, and a carefully designed retry policy.

In [ ]:
architecture = {
    "domain": "validated immutable events",
    "storage": "SQLite unique event ids and atomic aggregate updates",
    "interface": "explicit CLI or service adapter",
    "observability": "counts and stable error codes without sensitive payloads",
    "network_required": False,
}
print(architecture)
assert architecture["network_required"] is False

## Independent extensions

**Extension 1.** Add an explicit request schema version and compatible v1/v2 parsing.

In [ ]:
# TODO: implement v1/v2 parsing with tests.

**Extension 2.** Add a bounded async ingestion layer; ensure database operations don't block the event loop unexpectedly.

In [ ]:
# TODO: implement bounded ingestion with clear ownership and cancellation.

**Extension 3.** Export an audit summary of aggregate totals without disclosing individual raw payloads.

In [ ]:
# TODO: implement safe audit reporting.

**Extension 4.** Write an architecture decision record for SQLite vs a production database and queue.

In [ ]:
# TODO: record alternatives, rationale, and consequences.

## Final project checklist

- [ ] SQLite uniqueness enforces event identity.
- [ ] Insert and aggregate update occur in one transaction.
- [ ] Replayed payloads have no duplicate effect.
- [ ] Conflicts and malformed events receive safe status.
- [ ] Tests use independent in-memory database state.
- [ ] README documents how to run the packaged CLI.
- [ ] No real network activity or credentials are needed to run the notebook.